# Model 3: Body Shape Classification
**Dataset:** Body Measurement Dataset (Kaggle/UCI) + MediaPipe pose keypoints  
**Architecture:** Tabular classifier (XGBoost / RandomForest) từ body measurements  
**Output:** `body_shape_classifier.pkl`  
**Target:** Accuracy > 80%

---
**5 Body Shape Types:**
- **Hourglass**: bust ≈ hip, waist significantly smaller
- **Pear**: hip > bust, smaller shoulders
- **Apple**: bust > hip, wider midsection
- **Rectangle**: bust ≈ waist ≈ hip (similar measurements)
- **Inverted Triangle**: bust > hip, wider shoulders

**Dataset:** `https://www.kaggle.com/datasets/deadskull7/fer2013` → dùng body measurement dataset  
Hoặc: `https://www.kaggle.com/datasets/muratkokludataset/body-measurements-dataset`

In [ ]:
# ============================================================
# CELL 1: Install
# ============================================================
!pip install scikit-learn xgboost pandas numpy matplotlib seaborn pillow mediapipe opencv-python --quiet

import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
import xgboost as xgb

print("Libraries loaded.")

In [ ]:
# ============================================================
# CELL 2: Download & load dataset
# ============================================================
# Option A: Kaggle download
# !kaggle datasets download muratkokludataset/body-measurements-dataset -p ./data --unzip

# Option B: Tự tạo synthetic dataset nếu không có Kaggle API
# Dataset thực tế từ: https://www.kaggle.com/datasets/muratkokludataset/body-measurements-dataset

DATA_PATH = "/kaggle/input/body-measurements-dataset/body_measurements.csv"

if not os.path.exists(DATA_PATH):
    print("Dataset not found. Generating synthetic data for demonstration...")
    
    np.random.seed(42)
    n = 5000
    
    # Synthetic body measurements (đơn vị: cm)
    def gen_hourglass(n):
        bust = np.random.normal(90, 5, n)
        waist = bust - np.random.uniform(20, 30, n)
        hip = bust + np.random.uniform(-3, 3, n)
        height = np.random.normal(165, 7, n)
        weight = np.random.normal(60, 8, n)
        return pd.DataFrame({'bust': bust, 'waist': waist, 'hip': hip,
                              'height': height, 'weight': weight, 'label': 'Hourglass'})

    def gen_pear(n):
        bust = np.random.normal(83, 5, n)
        waist = np.random.normal(70, 4, n)
        hip = bust + np.random.uniform(12, 20, n)
        height = np.random.normal(163, 7, n)
        weight = np.random.normal(65, 9, n)
        return pd.DataFrame({'bust': bust, 'waist': waist, 'hip': hip,
                              'height': height, 'weight': weight, 'label': 'Pear'})

    def gen_apple(n):
        bust = np.random.normal(95, 6, n)
        waist = bust - np.random.uniform(5, 12, n)
        hip = bust - np.random.uniform(5, 10, n)
        height = np.random.normal(163, 7, n)
        weight = np.random.normal(75, 10, n)
        return pd.DataFrame({'bust': bust, 'waist': waist, 'hip': hip,
                              'height': height, 'weight': weight, 'label': 'Apple'})

    def gen_rectangle(n):
        bust = np.random.normal(85, 5, n)
        waist = bust - np.random.uniform(5, 12, n)
        hip = bust + np.random.uniform(-5, 5, n)
        height = np.random.normal(167, 7, n)
        weight = np.random.normal(58, 8, n)
        return pd.DataFrame({'bust': bust, 'waist': waist, 'hip': hip,
                              'height': height, 'weight': weight, 'label': 'Rectangle'})

    def gen_inverted(n):
        bust = np.random.normal(92, 5, n)
        waist = np.random.normal(75, 4, n)
        hip = bust - np.random.uniform(10, 18, n)
        height = np.random.normal(168, 7, n)
        weight = np.random.normal(63, 9, n)
        return pd.DataFrame({'bust': bust, 'waist': waist, 'hip': hip,
                              'height': height, 'weight': weight, 'label': 'InvertedTriangle'})

    df = pd.concat([
        gen_hourglass(n),
        gen_pear(n),
        gen_apple(n),
        gen_rectangle(n),
        gen_inverted(n)
    ], ignore_index=True)

    os.makedirs('./data', exist_ok=True)
    df.to_csv('./data/body_measurements.csv', index=False)
    DATA_PATH = './data/body_measurements.csv'
    print(f"Synthetic dataset created: {len(df)} samples")
else:
    df = pd.read_csv(DATA_PATH)

print(df.head())
print(f"\nShape: {df.shape}")
print("\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
# ============================================================
# CELL 3: Feature engineering
# ============================================================
df = df.dropna()

# Derived features - tỷ lệ quan trọng hơn số tuyệt đối
df['waist_hip_ratio']   = df['waist'] / df['hip']
df['bust_hip_ratio']    = df['bust']  / df['hip']
df['waist_bust_ratio']  = df['waist'] / df['bust']
df['bust_minus_hip']    = df['bust']  - df['hip']
df['hip_minus_waist']   = df['hip']   - df['waist']
df['bust_minus_waist']  = df['bust']  - df['waist']
df['bmi']               = df['weight'] / (df['height'] / 100) ** 2

FEATURES = ['bust', 'waist', 'hip', 'height', 'weight',
            'waist_hip_ratio', 'bust_hip_ratio', 'waist_bust_ratio',
            'bust_minus_hip', 'hip_minus_waist', 'bust_minus_waist', 'bmi']

X = df[FEATURES].values
y = df['label'].values

# Encode labels
le = LabelEncoder()
y_enc = le.fit_transform(y)
print("Classes:", le.classes_)

# Visualize pairplot của các tỷ lệ chính
plt.figure(figsize=(12, 4))
for i, (feat1, feat2) in enumerate([
    ('waist_hip_ratio', 'bust_hip_ratio'),
    ('bust_minus_waist', 'hip_minus_waist'),
    ('bmi', 'waist_hip_ratio')
]):
    plt.subplot(1, 3, i+1)
    for label in df['label'].unique():
        mask = df['label'] == label
        plt.scatter(df.loc[mask, feat1], df.loc[mask, feat2], 
                    alpha=0.3, s=5, label=label)
    plt.xlabel(feat1)
    plt.ylabel(feat2)
    if i == 0:
        plt.legend(markerscale=3, fontsize=7)
plt.tight_layout()
plt.savefig('body_shape_scatter.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 4: Train/test split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# ============================================================
# CELL 5: Train multiple models, compare
# ============================================================
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'XGBoost':      xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                                       use_label_encoder=False, eval_metric='mlogloss',
                                       random_state=42),
    'LogisticReg':  LogisticRegression(max_iter=1000, C=1.0),
    'SVM':          SVC(C=10, kernel='rbf', probability=True)
}

results = {}
for name, clf in models.items():
    clf.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, clf.predict(X_test_sc))
    cv  = cross_val_score(clf, X_train_sc, y_train, cv=5, scoring='accuracy').mean()
    results[name] = {'test_acc': acc, 'cv_acc': cv}
    print(f"{name:15s} | Test Acc: {acc:.4f} | CV Acc: {cv:.4f}")

# Pick best model
best_name = max(results, key=lambda k: results[k]['test_acc'])
best_model = models[best_name]
print(f"\nBest model: {best_name} ({results[best_name]['test_acc']*100:.1f}%)")

In [ ]:
# ============================================================
# CELL 6: Fine-tune best model (XGBoost)
# ============================================================
param_grid = {
    'n_estimators': [200, 300],
    'max_depth': [5, 6, 8],
    'learning_rate': [0.05, 0.1, 0.15]
}

xgb_model = xgb.XGBClassifier(
    use_label_encoder=False, eval_metric='mlogloss', random_state=42
)

grid = GridSearchCV(xgb_model, param_grid, cv=5, scoring='accuracy', 
                    n_jobs=-1, verbose=1)
grid.fit(X_train_sc, y_train)

print(f"Best params: {grid.best_params_}")
print(f"Best CV score: {grid.best_score_:.4f}")

final_model = grid.best_estimator_
final_acc   = accuracy_score(y_test, final_model.predict(X_test_sc))
print(f"Final Test Accuracy: {final_acc*100:.1f}%")

In [ ]:
# ============================================================
# CELL 7: Evaluation
# ============================================================
y_pred = final_model.predict(X_test_sc)
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, 
            yticklabels=le.classes_, cmap='Greens')
plt.title('Confusion Matrix - Body Shape Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_body.png', dpi=100)
plt.show()

# Feature importance
importances = final_model.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_df['feature'], feat_df['importance'], color='steelblue')
plt.title('Feature Importance - Body Shape')
plt.tight_layout()
plt.savefig('feature_importance_body.png', dpi=100)
plt.show()

In [ ]:
# ============================================================
# CELL 8: Save model + outfit recommendations map
# ============================================================
BODY_SHAPE_TIPS = {
    'Hourglass': {
        'recommended': ['wrap dresses', 'fitted blazers', 'high-waist jeans', 'bodycon'],
        'avoid': ['shapeless clothing', 'oversized tops'],
        'description': 'Balanced bust and hips with defined waist'
    },
    'Pear': {
        'recommended': ['A-line skirts', 'wide-leg pants', 'off-shoulder tops', 'boat neck'],
        'avoid': ['skinny jeans', 'hip pockets', 'pencil skirts'],
        'description': 'Hips wider than bust'
    },
    'Apple': {
        'recommended': ['empire waist', 'V-neck', 'flowy tops', 'wide-leg trousers'],
        'avoid': ['clingy fabrics around waist', 'cropped tops'],
        'description': 'Fuller midsection, slimmer legs'
    },
    'Rectangle': {
        'recommended': ['peplum tops', 'ruffle details', 'belted dresses', 'layered looks'],
        'avoid': ['straight column dresses', 'loose shapeless clothing'],
        'description': 'Similar bust, waist, and hip measurements'
    },
    'InvertedTriangle': {
        'recommended': ['A-line skirts', 'flared pants', 'V-neck', 'ruffled skirts'],
        'avoid': ['shoulder pads', 'boat neck', 'strapless tops'],
        'description': 'Broader shoulders than hips'
    }
}

with open('body_shape_classifier.pkl', 'wb') as f:
    pickle.dump({
        'model': final_model,
        'scaler': scaler,
        'label_encoder': le,
        'features': FEATURES,
        'body_shape_tips': BODY_SHAPE_TIPS,
        'val_acc': final_acc
    }, f)

print(f"Saved: body_shape_classifier.pkl")
print(f"Final accuracy: {final_acc*100:.1f}%")

In [ ]:
# ============================================================
# CELL 9: MediaPipe integration (Tasks API - mediapipe 0.10+)
# ============================================================
import urllib.request
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Download pose landmarker model (chỉ lần đầu)
MODEL_PATH = "./pose_landmarker_lite.task"
MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"

if not os.path.exists(MODEL_PATH):
    print("Downloading pose landmarker model (~5MB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print("Done.")

# Tạo detector
_base_opts = python.BaseOptions(model_asset_path=MODEL_PATH)
_pose_opts = vision.PoseLandmarkerOptions(base_options=_base_opts)
_detector  = vision.PoseLandmarker.create_from_options(_pose_opts)

# Landmark indices (giống PoseLandmark cũ)
_L_SHOULDER, _R_SHOULDER = 11, 12
_L_HIP,      _R_HIP      = 23, 24


def estimate_body_ratios_from_image(image_path):
    """
    Dùng MediaPipe Tasks API để ước tính số đo từ ảnh toàn thân.
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print(f"Không đọc được ảnh: {image_path}")
        return None

    h, w = img_bgr.shape[:2]
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    )
    result = _detector.detect(mp_image)

    if not result.pose_landmarks:
        print("Không phát hiện được pose.")
        return None

    lm = result.pose_landmarks[0]  # người đầu tiên

    shoulder_width = abs(lm[_L_SHOULDER].x - lm[_R_SHOULDER].x) * w
    hip_width      = abs(lm[_L_HIP].x      - lm[_R_HIP].x)      * w
    waist_width    = (shoulder_width + hip_width) / 2 * 0.85

    # Scale về cm (vai trung bình ~38cm)
    scale = 38.0 / shoulder_width if shoulder_width > 0 else 1.0
    bust  = shoulder_width * scale * 2.2
    waist = waist_width    * scale * 2.0
    hip   = hip_width      * scale * 2.3

    return {
        'bust':   round(bust, 1),
        'waist':  round(waist, 1),
        'hip':    round(hip, 1),
        'height': 165.0,   # cần nhập thủ công
        'weight': 60.0     # cần nhập thủ công
    }


def predict_body_shape_from_measurements(measurements, model_data):
    """Predict body shape từ dict số đo."""
    m = measurements
    features = [
        m['bust'], m['waist'], m['hip'], m['height'], m['weight'],
        m['waist'] / m['hip'],
        m['bust']  / m['hip'],
        m['waist'] / m['bust'],
        m['bust']  - m['hip'],
        m['hip']   - m['waist'],
        m['bust']  - m['waist'],
        m['weight'] / (m['height'] / 100) ** 2
    ]
    X     = model_data['scaler'].transform([features])
    pred  = model_data['model'].predict(X)[0]
    label = model_data['label_encoder'].inverse_transform([pred])[0]
    proba = model_data['model'].predict_proba(X)[0]
    tips  = model_data['body_shape_tips'].get(label, {})
    return {
        'body_shape':        label,
        'confidence':        float(proba.max()),
        'recommended_styles': tips.get('recommended', []),
        'avoid':             tips.get('avoid', []),
        'description':       tips.get('description', '')
    }


# Test với số đo thủ công
with open('body_shape_classifier.pkl', 'rb') as f:
    model_data = pickle.load(f)

test_measurements = {'bust': 90, 'waist': 68, 'hip': 92, 'height': 165, 'weight': 60}
result = predict_body_shape_from_measurements(test_measurements, model_data)
print("Body shape analysis result:")
print(json.dumps(result, indent=2, ensure_ascii=False))